# 13 — Delegation

Module 08 split Chinook into two specialists and a desk. That wiring is a product choice, not a law of nature.

**Most multi-agent systems should be one agent with more tools.** Extra agents earn their keep when the specialists need different tools, different permissions, or a different model. If they share a database and a prompt, you have bought a routing problem in exchange for a metaphor.

Today we put that claim on the same Helena facts (7 invoices, $49.62, Steve Johnson):

1. One agent, three tools. No desk.
2. Two specialists whose tool descriptions **overlap**. The desk has to pick a door.
3. The same two specialists with **sharp** descriptions.

CrewAI's role / goal / backstory idea gets a page, not an install. A2A vs MCP gets a table. Neither is a second lab.


## 1. Learn

```
01  who decides the next step
08  three wirings of two specialists
09  you draw the arrows
13  you are here — when a second agent is not the next step
```

A second agent is worth it when at least one of these is true:

| Reason | Example in this shop |
|---|---|
| Different tools | Invoices can read money. People cannot. |
| Different permissions | Invoices is read-only. Refunds (not today) would write. |
| Different model | A cheap router, a strong specialist. |
| Independent scale | Twenty invoice lookups, one people lookup. |

If none of those hold, one agent with the union of the tools is simpler, cheaper to debug, and does not need a router.

**Descriptions are the router.** Two tools that both say "look up a customer" are one tool with two names. A weak model will guess. A strong model may still guess. Sharpening the description is cheaper than adding a third agent.

This module only teaches the overlap failure **if this pin of nano still misses**. We will run it. If it names Steve anyway, that is data, not a broken cell. Re-verify before delivery. Prep on 2026-08-19: overlap 5/5 still named Steve.

**CrewAI** (role, goal, backstory) is a staffing metaphor for the same loop. We do not install it. It would downgrade `openai` and `mcp` in this venv.

**MCP vs A2A.** MCP is agent to tool. A2A is agent to agent. One table, no lab. Cut first.


## 2. Do

### Load Chinook and the three named tools

Same functions as 08. No free SQL. `check_same_thread=False` because the SDK runs tools on a worker thread.


In [1]:
from pathlib import Path
import os
import re
import sqlite3
import unicodedata

from dotenv import load_dotenv, find_dotenv
from agents import Agent, Runner, function_tool, trace, gen_trace_id


def fold(text: str) -> str:
    nfkd = unicodedata.normalize("NFKD", text)
    return "".join(ch for ch in nfkd if not unicodedata.combining(ch)).casefold()


load_dotenv(find_dotenv(usecwd=True))
ROOT = Path(find_dotenv(usecwd=True)).parent

api_key = os.environ.get("OPENAI_API_KEY", "").strip()
model = os.environ.get("MODEL_DEFAULT", "").strip()
model_strong = os.environ.get("MODEL_STRONG", "").strip() or model
assert api_key, "OPENAI_API_KEY is missing."
assert model, "MODEL_DEFAULT is missing."

db = sqlite3.connect(ROOT / "data" / "chinook.db", check_same_thread=False)
db.row_factory = sqlite3.Row

print("OPENAI_API_KEY is set:", True)
print("MODEL_DEFAULT:", model)
print("MODEL_STRONG:", model_strong)
print("customers:", db.execute("select count(*) from customers").fetchone()[0])


OPENAI_API_KEY is set: True
MODEL_DEFAULT: gpt-5.4-nano
MODEL_STRONG: gpt-5.4-mini
customers: 59


In [2]:
def find_customer(name: str):
    needle = fold(name).strip()
    rows = db.execute(
        "select CustomerId, FirstName, LastName, SupportRepId from customers"
    ).fetchall()
    hits = []
    for row in rows:
        full = fold(row["FirstName"] + " " + row["LastName"])
        if needle == full or needle == fold(row["FirstName"]) or needle in full:
            hits.append(row)
    if len(hits) == 1:
        return hits[0]
    if not hits:
        return None
    return "more than one customer matches that name"


def _customer_row(name: str):
    row = find_customer(name)
    if row is None:
        return None, "no customer by that name"
    if isinstance(row, str):
        return None, row
    return row, None


def count_invoices_fn(name: str) -> str:
    row, err = _customer_row(name)
    if err:
        return err
    n = db.execute(
        "select count(*) from invoices where CustomerId = ?",
        (row["CustomerId"],),
    ).fetchone()[0]
    return f"{row['FirstName']} {row['LastName']} has {n} invoices"


def invoice_total_fn(name: str) -> str:
    row, err = _customer_row(name)
    if err:
        return err
    total = db.execute(
        "select round(sum(Total), 2) from invoices where CustomerId = ?",
        (row["CustomerId"],),
    ).fetchone()[0]
    return f"{row['FirstName']} {row['LastName']} has spent {total} dollars"


def support_rep_fn(name: str) -> str:
    row, err = _customer_row(name)
    if err:
        return err
    rep = db.execute(
        "select FirstName || ' ' || LastName from employees where EmployeeId = ?",
        (row["SupportRepId"],),
    ).fetchone()[0]
    return f"{row['FirstName']} {row['LastName']}'s support rep is {rep}"


@function_tool
def count_invoices(name: str) -> str:
    "How many invoices this customer has."
    return count_invoices_fn(name)


@function_tool
def invoice_total(name: str) -> str:
    "Total amount this customer has spent, in dollars."
    return invoice_total_fn(name)


@function_tool
def support_rep(name: str) -> str:
    "The support representative for this customer."
    return support_rep_fn(name)


print(count_invoices_fn("Helena"))
print(invoice_total_fn("Helena Holy"))
print(support_rep_fn("Helena Holý"))


Helena Holý has 7 invoices
Helena Holý has spent 49.62 dollars
Helena Holý's support rep is Steve Johnson


Helena: **7**, **49.62**, **Steve Johnson**. No model yet.

### One agent, three tools

The default. One yellow box, three green ellipses. No router.


In [3]:
QUESTION = (
    "How many invoices does Helena Holý have, what did she spend, "
    "and who is her support rep?"
)

one = Agent(
    name="Desk",
    instructions="Use the tools. Do not invent numbers or names.",
    model=model,
    tools=[count_invoices, invoice_total, support_rep],
)

try:
    from agents.extensions.visualization import draw_graph
    draw_graph(one)
except Exception as exc:
    print("Graph not drawn:", exc)


In [4]:
trace_id = gen_trace_id()
print("Trace:", "https://platform.openai.com/traces/trace?trace_id=" + trace_id)
with trace("13 one agent", trace_id=trace_id):
    one_out = await Runner.run(one, QUESTION)
print(one_out.final_output)


Trace: https://platform.openai.com/traces/trace?trace_id=trace_c75ca01694e444f8bd900abaafc73f3a


Helena Holý has **7 invoices**, has spent **$49.62**, and her support rep is **Steve Johnson**.


All three facts from one process. That is the design you have to beat.

### Overlapping specialists

Two agents. Invoices owns count and spend. People owns the rep. Their **tool descriptions on the desk** are the same sentence: "Look up a customer." The desk is told to pick **one** door. The question is only the support-rep fact, so a wrong door cannot name Steve.


In [5]:
invoices = Agent(
    name="Invoices",
    instructions="Answer from your tools only. If you cannot, say you do not know.",
    model=model,
    tools=[count_invoices, invoice_total],
)
people = Agent(
    name="People",
    instructions="Answer from your tools only. If you cannot, say you do not know.",
    model=model,
    tools=[support_rep],
)

overlap = Agent(
    name="Desk",
    instructions="Pick exactly one specialist and use it. Do not invent names or numbers.",
    model=model,
    tools=[
        invoices.as_tool(tool_name="ask_invoices", tool_description="Look up a customer."),
        people.as_tool(tool_name="ask_people", tool_description="Look up a customer."),
    ],
)

WHO = "Who is Helena Holý's support representative?"
trace_id = gen_trace_id()
print("Trace:", "https://platform.openai.com/traces/trace?trace_id=" + trace_id)
with trace("13 overlap", trace_id=trace_id):
    overlap_out = await Runner.run(overlap, WHO)
print(overlap_out.final_output)
print()
print("Steve in the sentence:", "Steve" in (overlap_out.final_output or ""))


Trace: https://platform.openai.com/traces/trace?trace_id=trace_544a2b86ab6e437ca79c13c3bf4a3f3e


Helena Holý’s support representative is **Steve Johnson**.

Steve in the sentence: True


If the sentence names **Steve Johnson**, this pin of nano still picked `ask_people`. If it says it does not know, or talks about invoices, it opened the wrong door. Both outcomes are the lesson: **the description is the router.** We will not rerun until it fails.

### Sharp descriptions

Same two specialists. The desk still picks one door. The descriptions now say what each door cannot do.


In [6]:
sharp = Agent(
    name="Desk",
    instructions="Pick exactly one specialist and use it. Do not invent names or numbers.",
    model=model,
    tools=[
        invoices.as_tool(
            tool_name="ask_invoices",
            tool_description="Invoice count and total spend only. Cannot name a support representative.",
        ),
        people.as_tool(
            tool_name="ask_people",
            tool_description="Support representative name only. Cannot count invoices or spend.",
        ),
    ],
)

trace_id = gen_trace_id()
print("Trace:", "https://platform.openai.com/traces/trace?trace_id=" + trace_id)
with trace("13 sharp", trace_id=trace_id):
    sharp_out = await Runner.run(sharp, WHO)
print(sharp_out.final_output)
print()
print("Steve in the sentence:", "Steve" in (sharp_out.final_output or ""))


Trace: https://platform.openai.com/traces/trace?trace_id=trace_55ed076855a949c08b40ebac1a6b82f0


Helena Holý’s support representative is **Steve Johnson**.

Steve in the sentence: True


Sharpening text is cheaper than adding an agent.

### CrewAI, in one page

CrewAI would describe those two specialists as a **role**, a **goal**, and a **backstory**, then a task for the desk. That is a staffing metaphor for `Agent(instructions=..., tools=...)`. The loop underneath is still call-a-tool, then a sentence.

We do not install CrewAI. A dry-run into this venv downgrades `openai` and `mcp`, which is what modules 00-08 run on. Role / goal / backstory can live on a slide.

### MCP vs A2A

| | MCP (module 06) | A2A |
|---|---|---|
| Who talks | Agent to tool | Agent to agent |
| What you discover | Tool names and schemas | Agent cards and capabilities |
| What moves | tools/call | A task, handed across a process or a vendor |

If the other side is a function, that is MCP. If the other side is another loop with its own tools, that is A2A. We are not standing up an A2A server today.


## 3. Observe

Three runs. The one-agent cell used the three-part Helena question so you would trust the tools. Overlap and sharp used the one-fact question so a wrong door is visible.


In [7]:
def has_steve(text):
    return "Steve" in (text or "")


print(f"{'wiring':<22} {'Steve?':<8}  excerpt")
print(f"{'one agent (3-part)':<22} {'(n/a)':<8}  {(one_out.final_output or '')[:50]!r}")
print(f"{'overlap, one door':<22} {str(has_steve(overlap_out.final_output)):<8}  {(overlap_out.final_output or '')[:50]!r}")
print(f"{'sharp, one door':<22} {str(has_steve(sharp_out.final_output)):<8}  {(sharp_out.final_output or '')[:50]!r}")


wiring                 Steve?    excerpt
one agent (3-part)     (n/a)     'Helena Holý has **7 invoices**, has spent **$49.62'
overlap, one door      True      'Helena Holý’s support representative is **Steve Jo'
sharp, one door        True      'Helena Holý’s support representative is **Steve Jo'


Things to notice:

- One agent did not need a router. Three tools, three facts.
- Overlap made the two doors indistinguishable. If Steve still appeared, this pin held. Do not take that as a guarantee.
- Sharp descriptions are a prompt change. They are not a new framework.
- CrewAI would rename the same objects. A2A would put them on a wire. Neither fixes a vague tool_description.

## 4. Challenge

One agent, three tools. A new customer:

> How many invoices does Puja Srivastava have, what did she spend, and who is her support rep?

Bind `final_text`. The next cell checks for **6**, **36.64**, and **Jane**.


In [ ]:
# final_text = ...


In [ ]:
t = str(final_text)
assert re.search(r"\b6\b", t), "Puja has 6 invoices"
assert "36.64" in t, "Puja spent 36.64 dollars"
assert "Jane" in t, "Puja's support rep is Jane Peacock"
print("looks good")
